In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_kernels

In [ ]:
data = pd.read_csv("all_participants_data.csv", index_col="Participant")

indexes = data.index
participants = set(indexes)

targetValues = data["median_valence"]
data = data.drop(columns=['median_arousal', 'median_valence'])

scaler = StandardScaler()
scaledArray = scaler.fit_transform(data)
scaledData = pd.DataFrame(scaledArray, index=indexes)
scaledData["target"] = targetValues


    


In [ ]:
import numpy as np
import pandas as pd

def pairwise_transform_vectorized(df, feature_cols, target_col):

    X = df[feature_cols].to_numpy()
    y = df[target_col].to_numpy()
    n = len(df)

    # All (i,j) index pairs where i < j
    i, j = np.triu_indices(n, 1)

    # Skip ties
    mask = y[i] != y[j]
    i, j = i[mask], j[mask]

    # Vectorized labels: 1 if y[i] > y[j], else 0
    labels = (y[i] > y[j]).astype(np.int8)

    # Vectorized difference features
    X_diff = X[i] - X[j]     

    # Build output DataFrame
    out = pd.DataFrame(X_diff, columns=feature_cols)
    out["label"] = labels

    return out

In [ ]:
print(scaledData.columns.values.tolist())

In [ ]:
featureList = scaledData.columns.values.tolist()
featureList.remove("target")

pairwiseTransformations = []


for participant in participants:
    print(participant)
    filteredData = scaledData.loc[participant]

    pairwiseTransformations.append(pairwise_transform_vectorized(filteredData, featureList, "target"))




In [ ]:
#Logistic regression on the pairwise transformed data

from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression()

In [ ]:
#Neural network classifier with L hidden layers on the pairwise transformed data

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class PairwiseNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()

        layers = []
        last_dim = input_dim

        for _ in range(num_layers):
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            last_dim = hidden_dim

        # output layer for binary classification
        layers.append(nn.Linear(last_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
#LSTM classifier with K hidden layers on the pairwise transformed data


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        """
        input_size = 1 if reshaping each feature value into a single timestep
        hidden_size = LSTM hidden dimension
        num_layers = K LSTM layers
        """
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers == 1 else 0.2
        )
        
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)
    
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out_packed, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out_packed, batch_first=True)

        last = out[:, -1, :]
        return self.fc(last)

In [ ]:
from scipy.stats import pearsonr
import numpy as np

def CCcoefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

In [ ]:
from sklearn.linear_model import SGDClassifier

results = []
n = len(pairwiseTransformations)

for test_idx in range(n):
    print(f"\n=== LOPO fold (test participant {test_idx}) ===")

        # Test set
    test_df = pairwiseTransformations[test_idx]
    if len(test_df) == 0:
        results.append(np.nan)
        continue

        # Train set: all except test_idx 
    train_dfs = [pairwiseTransformations[i] for i in range(n) if i != test_idx]

        # 1) Incremental scaler
    scaler = StandardScaler()
    for df in train_dfs:
        X = df.drop(columns=["label"]).values
        if len(X) > 0:
            scaler.partial_fit(X)

        # 2) Incremental logistic regression
    clf = SGDClassifier(
        loss="log_loss",
        max_iter=1,
        tol=None,
        warm_start=True,
    )
    classes = np.array([0, 1])

        # 3) Train incrementally on each participant
    for df in train_dfs:
        X = df.drop(columns=["label"]).values
        y = df["label"].values
        if len(X) == 0:
            continue
        X = scaler.transform(X)
        clf.partial_fit(X, y, classes=classes)

        # 4) Evaluate
    X_test = scaler.transform(test_df.drop(columns=["label"]).values)
    y_test = test_df["label"].values
    y_pred = clf.predict_proba(X_test)[:, 1]

    r, _ = pearsonr(y_test, y_pred)
    ccc = CCcoefficient(y_test, y_pred)

    print(r, ccc)
    results.append((r, ccc))


print(results)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def lopo_cv_nn(
    pairwise_dfs,
    feature_cols,
    label_col="label",
    num_layers=2,
    hidden_dim=16,
    batch_size=4096,
    epochs=10,
    lr=1e-3
):
    results = []
    n = len(pairwise_dfs)

    input_dim = len(feature_cols)

    for test_idx in range(n):
        print(f"\n=== LOPO fold (test participant {test_idx}) ===")

        test_df = pairwise_dfs[test_idx]
        if len(test_df) == 0:
            results.append({"pearsonr": np.nan, "ccc": np.nan})
            continue

        # Build model
        model = PairwiseNN(input_dim, hidden_dim, num_layers).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.BCEWithLogitsLoss()

        # Training
        model.train()
        for df in [pairwise_dfs[i] for i in range(n) if i != test_idx]:

            X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
            y = torch.tensor(df[label_col].values, dtype=torch.float32).unsqueeze(1)

            dataset = TensorDataset(X, y)
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

            for epoch in range(epochs):
                for X_batch, y_batch in loader:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)

                    optimizer.zero_grad()
                    logits = model(X_batch)
                    loss = loss_fn(logits, y_batch)
                    loss.backward()
                    optimizer.step()

        model.eval()
        X_test = torch.tensor(test_df[feature_cols].values, dtype=torch.float32).to(device)
        y_test = test_df[label_col].values

        with torch.no_grad():
            preds = model(X_test).cpu().numpy().flatten()
            preds_prob = 1 / (1 + np.exp(-preds)) 

        r, _ = pearsonr(y_test, preds_prob)
        ccc = CCcoefficient(y_test, preds_prob)

        print(f"Fold {test_idx} → Pearson r = {r:.4f},  CCC = {ccc:.4f}")

        results.append((r, ccc))

    return results

In [ ]:
feature_cols = [c for c in pairwiseTransformations[0].columns if c != "label"]

results = lopo_cv_nn(
    pairwiseTransformations,
    feature_cols,
    num_layers=2,   
    hidden_dim=16,   
    batch_size=1024,  
    epochs=5,        
    lr=1e-3
)

print(results)